# **ILP Meal Planner**

This notebook tests the `ILPMealPlanner`, which uses Integer Linear Programming (via [PuLP](https://coin-or.github.io/pulp/) and the CBC solver) to find a provably near-optimal 7-day meal plan.

The ILP maximises the same composite objective as the GA:
$$\text{fitness} = \text{pantry\_score} - \text{dietary\_penalty} - \text{waste\_penalty} - \text{budget\_penalty}$$

It serves as an **oracle / upper bound** when evaluating other planners — any other planner's fitness score can be compared against the ILP to compute an *optimality gap*.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from pulp import LpStatus

from engines import ILPMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import (
    MealPlanningEnvironment,
    Pantry,
)


In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-30
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-03
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-05
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [12]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [13]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [14]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [15]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

## Solve

The ILP solver (CBC) will search for the optimal meal plan. The `time_limit` parameter caps the solve time; the best incumbent solution found within the limit is returned if the problem is not solved to proven optimality in time.

| Parameter | Description |
|---|---|
| `time_limit` | Maximum solver time in seconds (default 300) |
| `msg` | Print CBC solver log (`True` / `False`) |

In [16]:
planner = ILPMealPlanner(meal_planning_environment)

In [ ]:
best_meal_plan, fitness_score = planner.generate_meal_plan(time_limit=300, msg=True)

In [ ]:
print(f"Solve status : {LpStatus[planner.solve_status]}")
print(f"Fitness score: {fitness_score:.4f}")

Solve status : Optimal
Fitness score: 8.5324


In [ ]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,793.786648,6.213352,5d
1,broccoli,1500,1020.582833,479.417167,1d
2,rice,1000,384.266667,615.733333,6d


In [ ]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Cream of Broccoli Soup,Cream of Broccoli Soup,Olive Oil Couscous Cake with Crème Fraîche and...
1,2,Orange Chicken,Orange Chicken,Roast Turkey with Pesto-Rice Stuffing
2,3,Roast Turkey with Pesto-Rice Stuffing,Chicken Tortilla Soup,Orange Chicken
3,4,Orange Chicken,Orange Chicken,Roast Turkey with Pesto-Rice Stuffing
4,5,Orange Chicken,Roast Turkey with Pesto-Rice Stuffing,"Fried Rice with Ham, Egg, and Scallions"
5,6,"Fried Rice with Ham, Egg, and Scallions","Fried Rice with Ham, Egg, and Scallions",Roast Turkey with Pesto-Rice Stuffing
6,7,Steamed Broccoli with Olive Oil and Parmesan,Thyme-Toasted Almonds,Steamed Broccoli with Olive Oil and Parmesan


In [ ]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: \u20ac{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 56
Total cost: €54.61


,Ingredient,Quantity to Buy (g),Cost (€)
0,Available at Middle Eastern markets and from i...,12.5,0.12
1,Bacon slices,18.5,0.19
2,Crushed tortilla chips,25.0,0.25
3,Olive oil,18.5,0.19
4,Parmigiano-Reggiano,21.3,0.21
5,all purpose flour,15.0,0.15
6,baking powder,1.2,0.01
7,bay leaf,0.1,0.00
8,carrot,200.0,2.00
9,chicken breast,750.3,4.77


In [ ]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1974.0 kcal,51.3 g,-26.0 kcal,1.3 g
Day 2,1962.2 kcal,49.6 g,-37.8 kcal,-0.4 g
Day 3,2116.1 kcal,51.8 g,116.1 kcal,1.8 g
Day 4,1962.2 kcal,49.6 g,-37.8 kcal,-0.4 g
Day 5,2055.2 kcal,38.3 g,55.2 kcal,-11.7 g
Day 6,2148.2 kcal,27.0 g,148.2 kcal,-23.0 g
Day 7,1930.6 kcal,65.8 g,-69.4 kcal,15.8 g
